# 6부. 처음부터 끝까지 통합 실습

장별 완성 예제. 학습용 가상 데이터이며 공식 시험 제출 형식이 아니다. examples 폴더에서 전체 실행한다.

In [ ]:
from pathlib import Path
import os
if not Path('lab.py').is_file() and Path('../lab.py').is_file():
    os.chdir('..')
assert Path('lab.py').is_file(), 'Run with the examples folder and data present'
get_ipython().run_line_magic('matplotlib', 'inline')


## 26장. 분류 실습: 고객 이탈 예측

정형 가상 데이터로 탐색부터 최종 평가까지 연결한다.

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
candidates = {
    "logistic":LogisticRegression(max_iter=1000),
    "forest":RandomForestClassifier(n_estimators=80,max_depth=6,random_state=42,n_jobs=1),
}
scores = {}
for name, estimator in candidates.items():
    model = Pipeline([("prep",preprocessor()),("clf",estimator)]).fit(X_train,y_train)
    scores[name] = f1_score(y_val,model.predict(X_val),zero_division=0)
winner = max(scores,key=scores.get)
X_dev = pd.concat([X_train,X_val])
y_dev = pd.concat([y_train,y_val])
final = Pipeline([("prep",preprocessor()),("clf",candidates[winner])])
final.fit(X_dev,y_dev)
prediction = final.predict(X_test)
assert X_dev.index.equals(y_dev.index)
assert len(prediction) == len(y_test) == 120
print("Validation:",scores,"selected:",winner)
print(classification_report(y_test,prediction,zero_division=0))

## 27장. 회귀 실습: 서비스 이용량 예측

머신러닝·딥러닝을 같은 분할에서 비교한다.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from lab import split_task, preprocessor, keras_model, fit_dense, dense_predict, dataset

X_train, X_val, X_test, y_train, y_val, y_test = split_task("regression")
linear = Pipeline([("prep",preprocessor("regression")),("reg",LinearRegression())])
linear.fit(X_train,y_train)
prep = preprocessor("regression")
train = prep.fit_transform(X_train).astype("float32")
val = prep.transform(X_val).astype("float32")
mean, std = y_train.mean(), y_train.std(ddof=0)
assert std > 0
neural = keras_model(train.shape[1],"regression")
fit_dense(neural,train,((y_train-mean)/std).to_numpy(dtype="float32"),
          val,((y_val-mean)/std).to_numpy(dtype="float32"),epochs=12)
scores = {"linear":root_mean_squared_error(y_val,linear.predict(X_val)),
          "neural":root_mean_squared_error(y_val,dense_predict(neural,val)*std+mean)}
winner = min(scores,key=scores.get)
X_dev, y_dev = pd.concat([X_train,X_val]), pd.concat([y_train,y_val])
if winner == "linear":
    final = Pipeline([("prep",preprocessor("regression")),("reg",LinearRegression())])
    final.fit(X_dev,y_dev)
    prediction = final.predict(X_test)
else:
    prep_final = preprocessor("regression")
    dev = prep_final.fit_transform(X_dev).astype("float32")
    test = prep_final.transform(X_test).astype("float32")
    mean_final, std_final = y_dev.mean(), y_dev.std(ddof=0)
    final = keras_model(dev.shape[1],"regression")
    final.fit(dataset(dev,((y_dev-mean_final)/std_final).to_numpy(dtype="float32"),True),
              epochs=12,verbose=0)
    prediction = dense_predict(final,test)*std_final+mean_final
assert np.isfinite(prediction).all() and prediction.shape == y_test.shape
print("Validation RMSE:",scores,"selected:",winner)
print("Test RMSE hours:",root_mean_squared_error(y_test,prediction))

## 28장. 문제의 요구사항을 코드로 옮기기

변수명·조건·설정·출력을 체크 가능한 계약으로 바꾼다.

In [ ]:
from pathlib import Path
import tempfile
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from lab import split_task, preprocessor, load_customers

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
clf = Pipeline([("prep",preprocessor()),("model",LogisticRegression(max_iter=1000))])
clf.fit(X_train,y_train)
pred = clf.predict(X_val)
score = accuracy_score(y_val,pred)
output = pd.DataFrame({"customer_id":load_customers().loc[X_val.index,"customer_id"].to_numpy(),
                       "prediction":pred})
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory)/"predictions.csv"
    output.to_csv(path,index=False)
    saved = pd.read_csv(path)
    assert len(saved) == len(y_val)
    assert list(saved.columns) == ["customer_id","prediction"]
    assert saved["customer_id"].tolist() == output["customer_id"].tolist()
assert "churn" not in X_train.columns and "customer_id" not in X_train.columns
assert 0 <= score <= 1
print("Required outputs validated:",len(output),score)